# Inventaire des datasets

Ce notebook explore le dossier `dataset/` et résume, pour chaque dataset, les sous-dossiers disponibles, le type de données estimé et le nombre d'images.

In [2]:
from pathlib import Path
from collections import Counter
from html import escape

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from IPython.display import HTML, display
except ImportError:
    class HTML(str):
        pass

    def display(value):
        print(value)


DATASET_DIR = Path("dataset")
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".gif", ".webp"}

assert DATASET_DIR.exists(), f"Dossier introuvable: {DATASET_DIR.resolve()}"
DATASET_DIR.resolve()

PosixPath('/Users/megretleo/scr/2026/RetFound-Compar/dataset')

## Fonctions utilitaires

Ces fonctions ignorent les dossiers cachés et détectent les images à partir de leur extension.

In [3]:
def visible_dirs(path: Path):
    return sorted(
        [p for p in path.iterdir() if p.is_dir() and not p.name.startswith(".")],
        key=lambda p: p.name.lower(),
    )


def visible_files(path: Path):
    return [p for p in path.iterdir() if p.is_file() and not p.name.startswith(".")]


def is_image(path: Path) -> bool:
    return path.suffix.lower() in IMAGE_EXTENSIONS


def iter_visible_files(root: Path):
    for path in root.rglob("*"):
        if any(part.startswith(".") for part in path.relative_to(root).parts):
            continue
        if path.is_file():
            yield path


def iter_images(root: Path):
    for path in iter_visible_files(root):
        if is_image(path):
            yield path


def infer_data_type(dataset_name: str) -> str:
    name = dataset_name.lower()
    if "oct" in name:
        return "OCT rétinien"
    if any(token in name for token in ["aptos", "idrid", "messidor", "fundus", "papila", "jsiec", "retina"]):
        return "Images de fond d'œil / rétinographies"
    return "À vérifier"


def sort_rows(rows, columns):
    return sorted(rows, key=lambda row: tuple(str(row[column]).lower() for column in columns))


def show_table(rows):
    if not rows:
        display(HTML("<p>Aucune ligne à afficher.</p>"))
        return None

    if pd is not None:
        return pd.DataFrame(rows)

    columns = list(rows[0].keys())
    header = "".join(f"<th>{escape(column)}</th>" for column in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(row[column]))}</td>" for column in columns) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))
    return None

## Vue d'ensemble des datasets

Une ligne par sous-dossier direct de `dataset/`.

In [4]:
dataset_rows = []

for dataset_path in visible_dirs(DATASET_DIR):
    image_files = list(iter_images(dataset_path))
    all_files = list(iter_visible_files(dataset_path))
    direct_subdirs = visible_dirs(dataset_path)
    splits = [p.name for p in direct_subdirs if p.name.lower() in {"train", "val", "valid", "validation", "test"}]
    class_names = sorted(
        {
            class_dir.name
            for split_dir in direct_subdirs
            for class_dir in visible_dirs(split_dir)
        },
        key=str.lower,
    )
    extension_counts = Counter(p.suffix.lower() for p in image_files)

    dataset_rows.append(
        {
            "dataset": dataset_path.name,
            "type_donnees_estime": infer_data_type(dataset_path.name),
            "sous_dossiers_directs": ", ".join(p.name for p in direct_subdirs),
            "splits": ", ".join(splits) if splits else "-",
            "nb_classes": len(class_names),
            "nb_images": len(image_files),
            "extensions_images": ", ".join(f"{ext}: {n}" for ext, n in sorted(extension_counts.items())),
            "nb_fichiers_non_images": len(all_files) - len(image_files),
        }
    )

dataset_rows = sort_rows(dataset_rows, ["dataset"])
show_table(dataset_rows)

,dataset,type_donnees_estime,sous_dossiers_directs,splits,nb_classes,nb_images,extensions_images,nb_fichiers_non_images
0,APTOS2019,Images de fond d'œil / rétinographies,"test, train, val","test, train, val",5,3662,.png: 3662,1
1,Glaucoma_fundus,Images de fond d'œil / rétinographies,"test, train, val","test, train, val",3,1544,.png: 1544,1
2,IDRiD_data,Images de fond d'œil / rétinographies,"test, train, val","test, train, val",5,516,.png: 516,1
3,JSIEC,Images de fond d'œil / rétinographies,"test, train, val","test, train, val",39,1000,".jpg: 997, .tif: 3",1
4,MESSIDOR2,Images de fond d'œil / rétinographies,"test, train, val","test, train, val",5,1744,.png: 1744,1
5,OCTID,OCT rétinien,"test, train, val","test, train, val",5,572,.jpeg: 572,1
6,PAPILA,Images de fond d'œil / rétinographies,"test, train, val","test, train, val",3,488,.jpg: 488,1
7,Retina,Images de fond d'œil / rétinographies,"test, train, val","test, train, val",4,601,.png: 601,1


## Nombre d'images par split

Comptage des images dans les dossiers `train`, `val` et `test` quand ils existent.

In [5]:
split_rows = []

for dataset_path in visible_dirs(DATASET_DIR):
    for split_path in visible_dirs(dataset_path):
        if split_path.name.lower() not in {"train", "val", "valid", "validation", "test"}:
            continue
        split_rows.append(
            {
                "dataset": dataset_path.name,
                "split": split_path.name,
                "nb_images": sum(1 for _ in iter_images(split_path)),
                "nb_classes": len(visible_dirs(split_path)),
                "classes": ", ".join(p.name for p in visible_dirs(split_path)),
            }
        )

split_rows = sort_rows(split_rows, ["dataset", "split"])
show_table(split_rows)

,dataset,split,nb_images,nb_classes,classes
0,APTOS2019,test,1100,5,"anodr, bmilddr, cmoderatedr, dseveredr, eproli..."
1,APTOS2019,train,2048,5,"anodr, bmilddr, cmoderatedr, dseveredr, eproli..."
2,APTOS2019,val,514,5,"anodr, bmilddr, cmoderatedr, dseveredr, eproli..."
3,Glaucoma_fundus,test,465,3,"anormal_control, bearly_glaucoma, cadvanced_gl..."
4,Glaucoma_fundus,train,861,3,"anormal_control, bearly_glaucoma, cadvanced_gl..."
5,Glaucoma_fundus,val,218,3,"anormal_control, bearly_glaucoma, cadvanced_gl..."
6,IDRiD_data,test,103,5,"anoDR, bmildDR, cmoderateDR, dsevereDR, eproDR"
7,IDRiD_data,train,329,5,"anoDR, bmildDR, cmoderateDR, dsevereDR, eproDR"
8,IDRiD_data,val,84,5,"anoDR, bmildDR, cmoderateDR, dsevereDR, eproDR"
9,JSIEC,test,316,39,"0.0.Normal, 0.1.Tessellated fundus, 0.2.Large ..."


## Nombre d'images par classe

Détail par `dataset / split / classe`.

In [6]:
class_rows = []

for dataset_path in visible_dirs(DATASET_DIR):
    for split_path in visible_dirs(dataset_path):
        if split_path.name.lower() not in {"train", "val", "valid", "validation", "test"}:
            continue
        for class_path in visible_dirs(split_path):
            class_rows.append(
                {
                    "dataset": dataset_path.name,
                    "split": split_path.name,
                    "classe": class_path.name,
                    "nb_images": sum(1 for _ in iter_images(class_path)),
                }
            )

class_rows = sort_rows(class_rows, ["dataset", "split", "classe"])
show_table(class_rows)

,dataset,split,classe,nb_images
0,APTOS2019,test,anodr,542
1,APTOS2019,test,bmilddr,111
2,APTOS2019,test,cmoderatedr,300
3,APTOS2019,test,dseveredr,58
4,APTOS2019,test,eproliferativedr,89
...,...,...,...,...
202,Retina,train,ddretina_disease,56
203,Retina,val,anormal,42
204,Retina,val,bcataract,14
205,Retina,val,cglaucoma,14


## Totaux rapides

Résumé global du dossier `dataset/`.

In [7]:
total_images = sum(row["nb_images"] for row in dataset_rows)
total_datasets = len(dataset_rows)

print(f"Nombre de datasets: {total_datasets}")
print(f"Nombre total d'images: {total_images}")

ranked_datasets = sorted(
    [
        {
            "dataset": row["dataset"],
            "type_donnees_estime": row["type_donnees_estime"],
            "nb_images": row["nb_images"],
        }
        for row in dataset_rows
    ],
    key=lambda row: row["nb_images"],
    reverse=True,
)
show_table(ranked_datasets)

Nombre de datasets: 8
Nombre total d'images: 10127


,dataset,type_donnees_estime,nb_images
0,APTOS2019,Images de fond d'œil / rétinographies,3662
1,MESSIDOR2,Images de fond d'œil / rétinographies,1744
2,Glaucoma_fundus,Images de fond d'œil / rétinographies,1544
3,JSIEC,Images de fond d'œil / rétinographies,1000
4,Retina,Images de fond d'œil / rétinographies,601
5,OCTID,OCT rétinien,572
6,IDRiD_data,Images de fond d'œil / rétinographies,516
7,PAPILA,Images de fond d'œil / rétinographies,488


In [8]:
from pathlib import Path

DATASET_DIR = Path("dataset")
SPLITS = {"train"}

def visible_dirs(path: Path):
    return sorted(
        [p for p in path.iterdir() if p.is_dir() and not p.name.startswith(".")],
        key=lambda p: p.name.lower(),
    )

dataset_classes = {}

for dataset_path in visible_dirs(DATASET_DIR):
    classes = set()

    for split_path in visible_dirs(dataset_path):
        if split_path.name.lower() not in SPLITS:
            continue

        for class_path in visible_dirs(split_path):
            classes.add(class_path.name)

    dataset_classes[dataset_path.name] = sorted(classes, key=str.lower)

dataset_classes


{'APTOS2019': ['anodr',
  'bmilddr',
  'cmoderatedr',
  'dseveredr',
  'eproliferativedr'],
 'Glaucoma_fundus': ['anormal_control',
  'bearly_glaucoma',
  'cadvanced_glaucoma'],
 'IDRiD_data': ['anoDR', 'bmildDR', 'cmoderateDR', 'dsevereDR', 'eproDR'],
 'JSIEC': ['0.0.Normal',
  '0.1.Tessellated fundus',
  '0.2.Large optic cup',
  '0.3.DR1',
  '1.0.DR2',
  '1.1.DR3',
  '10.0.Possible glaucoma',
  '10.1.Optic atrophy',
  '11.Severe hypertensive retinopathy',
  '12.Disc swelling and elevation',
  '13.Dragged Disc',
  '14.Congenital disc abnormality',
  '15.0.Retinitis pigmentosa',
  '15.1.Bietti crystalline dystrophy',
  '16.Peripheral retinal degeneration and break',
  '17.Myelinated nerve fiber',
  '18.Vitreous particles',
  '19.Fundus neoplasm',
  '2.0.BRVO',
  '2.1.CRVO',
  '20.Massive hard exudates',
  '21.Yellow-white spots-flecks',
  '22.Cotton-wool spots',
  '23.Vessel tortuosity',
  '24.Chorioretinal atrophy-coloboma',
  '25.Preretinal hemorrhage',
  '26.Fibrosis',
  '27.Laser S